In [89]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [2]:
spark=SparkSession.builder.appName("Customer Churn Analysis and Retention Strategy").master('yarn').getOrCreate()

**Extract Data From Kaggle and load into HDFS**

**Step1**: **Download** data from **kaggle** inside the **jupyter container**

**Step2**: **Copy** the data to the **namenode container**

**Step3**: **Put** the data inside **HDFS**FS**

In [3]:
#!/bin/bash
#curl -L -o /churn_data/synthetic-telco-messy-dataset-churn-prediction.zip\
#https://www.kaggle.com/api/v1/datasets/download/marwanehosni/synthetic-telco-messy-dataset-churn-prediction#
#unzip /churn_data/synthetic-telco-messy-dataset-churn-prediction.zip -d /churn_data
#hadoop fs -put /churn_data/synthetic-telco-messy-dataset-churn-prediction.zip /churn_data/

**Load Data By Pyspark Yo Start cleaning**

In [4]:
hdfs_path='/churn_data/telco_customer_data_v2.csv'
churn_schema='''customerID string , gender string,SeniorCitizen string , Partner string  , Depndents string , tenure float, PhoneService string, MultipleLines string,
InternetService string, OnlineSecurity string, OnlineBackup string, DeviceProtection string, TechSupport string, StreamingTV string, StreamingMovies string, 
Contract string, PaperlessBilling string ,PaymentMethod string, MonthlyCharges float, TotalCharges float , Churn string'''


In [5]:
churn_data=spark.read.csv(hdfs_path,header=True , schema=churn_schema)
churn_data.cache()
churn_data.select(churn_data.columns[0:5]).show(5)
churn_data.select(churn_data.columns[5:10]).show(5)
churn_data.select(churn_data.columns[10:15]).show(5)
churn_data.select(churn_data.columns[15:22]).show(5)

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
| CUST00001|  Male|            0|     No|      Yes|
| CUST00002|  Male|            1|    Yes|       No|
| CUST00003|Female|           No|     No|       No|
| CUST00004|Female|            0|     No|      Yes|
| CUST00005|  Male|          Yes|    Yes|      Yes|
+----------+------+-------------+-------+---------+
only showing top 5 rows

+------+------------+-------------+---------------+-------------------+
|tenure|PhoneService|MultipleLines|InternetService|     OnlineSecurity|
+------+------------+-------------+---------------+-------------------+
|   3.0|         Yes|          Yes|             No|No internet service|
|   2.0|         Yes|          Yes|            DSL|                 No|
|  42.0|         Yes|          Yes|            DSL|                 No|
|  40.0|         Yes|          Yes|    Fiber optic|                 No|
|  

**Check For Duplicates**

In [6]:
churn_data.groupBy(churn_data.columns[0]).count().filter('count>1').show()

+----------+-----+
|customerID|count|
+----------+-----+
+----------+-----+



**Check For Nulls**

In [7]:
null_data=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])

In [8]:
null_data.select(null_data.columns[0:5]).show()
null_data.select(null_data.columns[5:10]).show()
null_data.select(null_data.columns[10:15]).show()
null_data.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|   748|          659|   3530|     3565|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|   567|           0|         1868|              0|          2922|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|        2747|            2894|       2733|       2827|           2785|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

**Handle Data Quality then Handling Missing Values**

**COLUMNS-> gender,SeniorCitizen,Partner,Depndents**



In [9]:
churn_data.groupBy('gender').count().show()

+------+-----+
|gender|count|
+------+-----+
|   Man|  500|
|     m|  339|
|  NULL|  748|
|     f|  326|
|Female|34110|
|  male|  374|
|  Male|33252|
|FEMALE|  351|
+------+-----+



In [10]:
churn_data=churn_data.withColumn('gender',when(col('gender')=='Man','Male').when(col('gender')=='m','Male')\
                                .when(col('gender')=='male','Male').when(col('gender')=='Male','Male')\
                                            .when(col('gender')=='f','Female').when(col('gender')=='FEMALE','Female').when(col('gender')=='Female','Female'))

In [11]:
churn_data.groupBy('gender').count().show()
churn_data=churn_data.fillna({'gender':'Unknown'})


+------+-----+
|gender|count|
+------+-----+
|  NULL|  748|
|Female|34787|
|  Male|34465|
+------+-----+



In [12]:
churn_data.groupBy('SeniorCitizen').count().show()

+-------------+-----+
|SeniorCitizen|count|
+-------------+-----+
|            0|52405|
|         NULL|  659|
|           No| 2739|
|          Yes| 3559|
|            1|10388|
|   not senior|  250|
+-------------+-----+



In [13]:
churn_data=churn_data.withColumn('SeniorCitizen',when(col('SeniorCitizen').isin('0','not senior','No' ),'No')
                                            .when(col('SeniorCitizen').isin('1','Yes'),'Yes'))
churn_data=churn_data.fillna({'SeniorCitizen':'Unknown'})

In [14]:
churn_data.groupBy('SeniorCitizen').count().show()

+-------------+-----+
|SeniorCitizen|count|
+-------------+-----+
|      Unknown|  659|
|           No|55394|
|          Yes|13947|
+-------------+-----+



In [15]:
churn_data.groupBy('Partner').count().show()

+-------+-----+
|Partner|count|
+-------+-----+
|   NULL| 3530|
|     No|38344|
|    Yes|28126|
+-------+-----+



In [16]:
churn_data=churn_data.fillna({'Partner':'Unknown'})
churn_data.groupBy('Partner').count().show()

+-------+-----+
|Partner|count|
+-------+-----+
|Unknown| 3530|
|     No|38344|
|    Yes|28126|
+-------+-----+



In [17]:
churn_data.groupBy('Depndents').count().show()

+---------+-----+
|Depndents|count|
+---------+-----+
|     NULL| 3565|
|       No|45375|
|      Yes|21060|
+---------+-----+



In [18]:
churn_data=churn_data.fillna({'Depndents':'Unknow'})
churn_data.groupBy('Depndents').count().show()

+---------+-----+
|Depndents|count|
+---------+-----+
|       No|45375|
|      Yes|21060|
|   Unknow| 3565|
+---------+-----+



**COLUMNS ====> PhoneService,MultipleLines,OnlineSecurity**

In [19]:
null_data2=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])
null_data2.select(null_data.columns[0:5]).show()
null_data2.select(null_data.columns[5:10]).show()
null_data2.select(null_data.columns[10:15]).show()
null_data2.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|     0|            0|      0|        0|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|   567|           0|         1868|              0|          2922|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|        2747|            2894|       2733|       2827|           2785|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

In [20]:
churn_data.groupBy('MultipleLines').count().show()

+----------------+-----+
|   MultipleLines|count|
+----------------+-----+
|No phone service| 8259|
|            NULL| 1868|
|              No|25192|
|             Yes|34681|
+----------------+-----+



In [21]:
churn_data=churn_data.withColumn('MultipleLines',when(col('MultipleLines').isin('No phone service','No'),'No').when(col('MultipleLines')=='Yes','Yes').otherwise('Unknown'))

In [22]:
churn_data.groupBy('MultipleLines').count().show()

+-------------+-----+
|MultipleLines|count|
+-------------+-----+
|      Unknown| 1868|
|           No|33451|
|          Yes|34681|
+-------------+-----+



In [23]:
churn_data.groupBy('OnlineSecurity').count().show()

+-------------------+-----+
|     OnlineSecurity|count|
+-------------------+-----+
|               NULL| 2922|
|                  Y|    3|
|                 No|39195|
|                Yes|11202|
|               True|    6|
|No internet service|16672|
+-------------------+-----+



In [24]:
churn_data=churn_data.withColumn('OnlineSecurity',when(col('OnlineSecurity').isin('Y','Yes','True'),'Yes') \
                                            .when(col('OnlineSecurity').isin('No','No internet service'),'No').otherwise('Unknown'))

In [25]:
churn_data.groupBy('OnlineSecurity').count().show()

+--------------+-----+
|OnlineSecurity|count|
+--------------+-----+
|       Unknown| 2922|
|            No|55867|
|           Yes|11211|
+--------------+-----+



**COLUMNS ====> OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies**

In [26]:
null_data3=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])
null_data3.select(null_data.columns[0:5]).show()
null_data3.select(null_data.columns[5:10]).show()
null_data3.select(null_data.columns[10:15]).show()
null_data3.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|     0|            0|      0|        0|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|   567|           0|            0|              0|             0|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|        2747|            2894|       2733|       2827|           2785|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

In [27]:
churn_data.groupBy('OnlineBackup').count().show()

+-------------------+-----+
|       OnlineBackup|count|
+-------------------+-----+
|               NULL| 2747|
|                  Y|   10|
|                 No|39308|
|                Yes|11237|
|               True|    7|
|No internet service|16691|
+-------------------+-----+



In [28]:
churn_data=churn_data.withColumn('OnlineBackup',when(col('OnlineBackup').isin('Y','Yes','True'),'Yes').when(col('OnlineBackup').isin('No','No internet service'),'Now')\
                                          .otherwise('Unknown'))

In [29]:
churn_data=churn_data.withColumn('OnlineBackup',when(col('OnlineBackup')=='Yes','Yes').when(col('OnlineBackup')=='Now','No')\
                                          .otherwise('Unknown'))
churn_data.groupBy('OnlineBackup').count().show()

+------------+-----+
|OnlineBackup|count|
+------------+-----+
|     Unknown| 2747|
|          No|55999|
|         Yes|11254|
+------------+-----+



In [30]:
churn_data.groupBy('DeviceProtection').count().show()

+-------------------+-----+
|   DeviceProtection|count|
+-------------------+-----+
|               NULL| 2894|
|                  Y|    6|
|                 No|39199|
|                Yes|11277|
|No internet service|16623|
|               True|    1|
+-------------------+-----+



In [31]:
churn_data=churn_data.withColumn('DeviceProtection',when(col('DeviceProtection').isin('Y','Yes','True'),'Yes').when(col('DeviceProtection').isin('No','No internet service'),'No')\
                                          .otherwise('Unknown'))

In [32]:
churn_data.groupBy('DeviceProtection').count().show()

+----------------+-----+
|DeviceProtection|count|
+----------------+-----+
|         Unknown| 2894|
|              No|55822|
|             Yes|11284|
+----------------+-----+



In [33]:
churn_data.groupBy('TechSupport').count().show()

+-------------------+-----+
|        TechSupport|count|
+-------------------+-----+
|               NULL| 2733|
|                  Y|    7|
|                 No|39401|
|                Yes|11239|
|               True|    4|
|No internet service|16616|
+-------------------+-----+



In [34]:
churn_data=churn_data.withColumn('TechSupport',when(col('TechSupport').isin('Y','Yes','True'),'Yes').when(col('TechSupport').isin('No','No internet service'),'No')\
                                          .otherwise('Unknown'))

In [35]:
churn_data.groupBy('TechSupport').count().show()

+-----------+-----+
|TechSupport|count|
+-----------+-----+
|    Unknown| 2733|
|         No|56017|
|        Yes|11250|
+-----------+-----+



In [36]:
churn_data.groupBy('StreamingTV').count().show()

+-------------------+-----+
|        StreamingTV|count|
+-------------------+-----+
|               NULL| 2827|
|                  Y|    5|
|                 No|39198|
|                Yes|11355|
|               True|    5|
|No internet service|16610|
+-------------------+-----+



In [37]:
churn_data=churn_data.withColumn('StreamingTV',when(col('StreamingTV').isin('Y','Yes','True'),'Yes').when(col('StreamingTV').isin('No','No internet service'),'No')\
                                          .otherwise('Unknown'))

In [38]:
churn_data.groupBy('StreamingTV').count().show()

+-----------+-----+
|StreamingTV|count|
+-----------+-----+
|    Unknown| 2827|
|         No|55808|
|        Yes|11365|
+-----------+-----+



In [39]:
churn_data.groupBy('StreamingMovies').count().show()

+-------------------+-----+
|    StreamingMovies|count|
+-------------------+-----+
|               NULL| 2785|
|                  Y|    7|
|                 No|39189|
|                Yes|11271|
|               True|    4|
|No internet service|16744|
+-------------------+-----+



In [40]:
churn_data=churn_data.withColumn('StreamingMovies',when(col('StreamingMovies').isin('Y','Yes','True'),'Yes').when(col('StreamingMovies').isin('No','No internet service'),'No')\
                                          .otherwise('Unknown'))

In [41]:
churn_data.groupBy('StreamingMovies').count().show()

+---------------+-----+
|StreamingMovies|count|
+---------------+-----+
|        Unknown| 2785|
|             No|55933|
|            Yes|11282|
+---------------+-----+



In [42]:
churn_data.groupBy('Contract').count().show()

+--------------+-----+
|      Contract|count|
+--------------+-----+
|Month-to-month|41845|
|      One year|14015|
|      Two year|13974|
|           M-M|  166|
+--------------+-----+



In [43]:
churn_data=churn_data.withColumn('Contract',when(col('Contract').isin('Month-to-month','M-M'),'Monthly')\
                                       .when(col('Contract').isin('One year'),'One Year') \
                                        .when(col('Contract').isin('Two year'),'Two year').otherwise('Unknown'))

In [44]:
churn_data.groupBy('Contract').count().show()

+--------+-----+
|Contract|count|
+--------+-----+
|One Year|14015|
|Two year|13974|
| Monthly|42011|
+--------+-----+



In [45]:
churn_data.groupBy('Churn').count().show()

+--------+-----+
|   Churn|count|
+--------+-----+
| CHURNED|   46|
|       Y|   39|
| Unknown|   54|
|       N|   46|
|      No|37057|
|     Yes|32713|
|NO CHURN|   45|
+--------+-----+



In [46]:
churn_data=churn_data.withColumn('Churn',when(col('Churn').isin('CHURNED','Y','Yes'),'Yes')\
                                       .when(col('Churn').isin('N','No','NO CHURN'),'No') \
                                        .otherwise('Unknown'))

In [47]:
churn_data.groupBy('Churn').count().show()

+-------+-----+
|  Churn|count|
+-------+-----+
|Unknown|   54|
|     No|37148|
|    Yes|32798|
+-------+-----+



**COLUMNS ====> PaymentMethod,MonthlyCharges,TotalCharges**

In [48]:
null_data4=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])
null_data4.select(null_data.columns[0:5]).show()
null_data4.select(null_data.columns[5:10]).show()
null_data4.select(null_data.columns[10:15]).show()
null_data4.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|     0|            0|      0|        0|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|   567|           0|            0|              0|             0|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|           0|               0|          0|          0|              0|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

In [49]:
churn_data.groupBy('PaymentMethod').count().show()

+--------------------+-----+
|       PaymentMethod|count|
+--------------------+-----+
|Credit card (auto...|10400|
|       BANK TRANSFER|  125|
|                NULL| 3569|
|        Mailed check|13963|
|Bank transfer (au...|14102|
|    Electronic check|27841|
+--------------------+-----+



In [50]:
churn_data=churn_data.fillna({'PaymentMethod':'Unknown'})
churn_data.groupBy('PaymentMethod').count().show()

+--------------------+-----+
|       PaymentMethod|count|
+--------------------+-----+
|Credit card (auto...|10400|
|       BANK TRANSFER|  125|
|        Mailed check|13963|
|             Unknown| 3569|
|Bank transfer (au...|14102|
|    Electronic check|27841|
+--------------------+-----+



**HANDLING NUMERICAL COLUMNS**

**COLUMNS ====> tenure,MonthlyCharges,TotalCharges**

In [51]:
null_data5=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])
null_data5.select(null_data.columns[0:5]).show()
null_data5.select(null_data.columns[5:10]).show()
null_data5.select(null_data.columns[10:15]).show()
null_data5.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|     0|            0|      0|        0|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|   567|           0|            0|              0|             0|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|           0|               0|          0|          0|              0|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

In [68]:
from pyspark.ml.feature import Imputer
## TO GET THE MEDIAN VALUE OF THE DATA TO FILL IT WITH THE MISSING VALUE
## Found negative values 

In [56]:
churn_data=churn_data.withColumn('tenure',abs(col('tenure'))).withColumn('MonthlyCharges',abs(col('MonthlyCharges')))

In [57]:
imputer=Imputer(inputCols=['tenure','MonthlyCharges'],outputCols=['tenure','MonthlyCharges']).setStrategy('median')
churn_data=imputer.fit(churn_data).transform(churn_data)

In [58]:
churn_data=churn_data.withColumn('TotalCharges',(col('tenure')*col('MonthlyCharges')))

In [63]:
null_data4=churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns])
null_data4.select(null_data.columns[0:5]).show()
null_data4.select(null_data.columns[5:10]).show()
null_data4.select(null_data.columns[10:15]).show()
null_data4.select(null_data.columns[15:25]).show()

+----------+------+-------------+-------+---------+
|customerID|gender|SeniorCitizen|Partner|Depndents|
+----------+------+-------------+-------+---------+
|         0|     0|            0|      0|        0|
+----------+------+-------------+-------+---------+

+------+------------+-------------+---------------+--------------+
|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|
+------+------------+-------------+---------------+--------------+
|     0|           0|            0|              0|             0|
+------+------------+-------------+---------------+--------------+

+------------+----------------+-----------+-----------+---------------+
|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|
+------------+----------------+-----------+-----------+---------------+
|           0|               0|          0|          0|              0|
+------------+----------------+-----------+-----------+---------------+

+--------+----------------+-------------+-

In [66]:
def check_quality(df):
    for x in range(21):
        df.groupBy(df.columns[x]).count().show(5)

In [67]:
check_quality(churn_data)

+----------+-----+
|customerID|count|
+----------+-----+
| CUST00050|    1|
| CUST00166|    1|
| CUST00437|    1|
| CUST00854|    1|
| CUST00865|    1|
+----------+-----+
only showing top 5 rows

+-------+-----+
| gender|count|
+-------+-----+
| Female|34787|
|Unknown|  748|
|   Male|34465|
+-------+-----+

+-------------+-----+
|SeniorCitizen|count|
+-------------+-----+
|      Unknown|  659|
|           No|55394|
|          Yes|13947|
+-------------+-----+

+-------+-----+
|Partner|count|
+-------+-----+
|Unknown| 3530|
|     No|38344|
|    Yes|28126|
+-------+-----+

+---------+-----+
|Depndents|count|
+---------+-----+
|       No|45375|
|      Yes|21060|
|   Unknow| 3565|
+---------+-----+

+------+-----+
|tenure|count|
+------+-----+
|  18.0| 1636|
|  64.0|  101|
|  47.0|  990|
|   9.0| 1738|
|  58.0|  108|
+------+-----+
only showing top 5 rows

+------------+-----+
|PhoneService|count|
+------------+-----+
|          No| 6989|
|         Yes|63011|
+------------+-----+

+--------

In [70]:
churn_data.select([count(when(col(c).isNull(),1)).alias(c) for c in churn_data.columns]).show()

+----------+------+-------------+-------+---------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Depndents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|Contract|PaperlessBilling|PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+---------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------+----------------+-------------+--------------+------------+-----+
|         0|     0|            0|      0|        0|     0|           0|            0|              0|             0|           0|               0|          0|          0|              0|       0|               0|     

**SAVING THE DATA FOR THE SERVING LAYERS**

In [72]:
churn_data.write.saveAsTable('churn_data')

In [114]:
churn_data.write.mode('overwrite').csv('/churn_output/',header=True)

In [76]:
spark.sql('show tables').show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|  default|churn_data|      false|
+---------+----------+-----------+



**STAR SCHEMA CREATION**

In [95]:
dim_customer=churn_data.select('customerID','gender','SeniorCitizen','Partner','Depndents','Churn')
dim_services=churn_data.select('PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies')
dim_contract=churn_data.select('Contract','PaperlessBilling','PaymentMethod')

In [96]:
dim_services=dim_services.withColumn('service_id',row_number().over(Window.orderBy(monotonically_increasing_id())) - 1)
dim_contract=dim_contract.withColumn('contract_id',row_number().over(Window.orderBy(monotonically_increasing_id())) - 1)

In [105]:
fact_table=churn_data.select('customerID','tenure','MonthlyCharges','TotalCharges')

In [107]:
fact_table=fact_table.withColumn('fact_id',row_number().over(Window.orderBy(monotonically_increasing_id())) - 1)
fact_table=fact_table.withColumn('service_id',row_number().over(Window.orderBy(monotonically_increasing_id())) - 1) \
                        .withColumn('contract_id',row_number().over(Window.orderBy(monotonically_increasing_id())) - 1)

In [109]:
fact_table.show(5)
dim_services.show(5)
dim_contract.show(5)
dim_customer.show(5)

+----------+------+--------------+------------+-------+----------+-----------+
|customerID|tenure|MonthlyCharges|TotalCharges|fact_id|service_id|contract_id|
+----------+------+--------------+------------+-------+----------+-----------+
| CUST00001|   3.0|         68.61|      205.83|      0|         0|          0|
| CUST00002|   2.0|         23.15|        46.3|      1|         1|          1|
| CUST00003|  42.0|         42.63|   1790.4601|      2|         2|          2|
| CUST00004|  40.0|         75.04|      3001.6|      3|         3|          3|
| CUST00005|  17.0|         22.38|      380.46|      4|         4|          4|
+----------+------+--------------+------------+-------+----------+-----------+
only showing top 5 rows

+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+----------+
|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingM

In [121]:
output_path='/churn_star_schema/'
fact_table.write.mode('overwrite').csv(output_path+'fact/',header=True)

In [122]:
dim_customer.write.mode('overwrite').csv(output_path+'customer/',header=True)
dim_services.write.mode('overwrite').csv(output_path+'services/',header=True)
dim_contract.write.mode('overwrite').csv(output_path+'contract/',header=True)